# LAB 09 — Declarative Automation Bundles: Deploy RetailHub

**Goal:** deploy the RetailHub capstone (Lakeflow Spark Declarative Pipeline + 3-task Lakeflow Job) to a `dev` target as a **Declarative Automation Bundle** (formerly *Databricks Asset Bundles / DABs*), and verify the deployment programmatically.

**Prerequisites:**
- lab_07 completed (the pipeline sources in `materials/lakeflow/lakeflow_demo` are familiar)
- The repo available as a **Git folder** in your workspace
- CLI access for tasks 3 & 5 — **local Databricks CLI** (or the web terminal, if your workspace offers it). No CLI? Task 3 also lists a **workspace UI** option and a **trainer-driven fallback**.

| Task | What you do | Where |
|------|-------------|-------|
| 1 | Inspect `databricks.yml` — bundle anatomy | notebook |
| 2 | Compose the deploy command with your catalog variable | notebook |
| *opt.* | Git folder: branch, commit & push, pull request (trainer-dependent) | UI |
| 3 | `databricks bundle validate` / `deploy`, then record how you ran it | terminal (or UI / trainer) + notebook |
| 4 | Verify the deployment with the Databricks SDK | notebook |
| 5 | Run the job and confirm a terminal SUCCESS state | terminal or UI + notebook |
| 6 | Reflection — `dev` vs `prod` targets | notebook |

## Setup

In [0]:
%run ../../setup/00_setup

In [0]:
# PROVIDED — run as-is (no changes needed)
import os

# The bundle lives in materials/cicd/ at the repo root. Walk up from this notebook's folder
# until materials/cicd/databricks.yml is found — works from notebooks/day3/lab/ and notebooks/solution/.
def find_bundle_dir(start=None):
    d = os.path.abspath(start or os.getcwd())
    while True:
        candidate = os.path.join(d, "materials", "cicd")
        if os.path.isfile(os.path.join(candidate, "databricks.yml")):
            return candidate
        parent = os.path.dirname(d)
        if parent == d:          # reached the filesystem root
            return None
        d = parent

BUNDLE_DIR = find_bundle_dir()
print(f"Bundle root: {BUNDLE_DIR}\n")

try:
    if BUNDLE_DIR is None:
        raise FileNotFoundError(f"materials/cicd/databricks.yml not found in {os.getcwd()} or any parent folder")
    databricks_yml = open(os.path.join(BUNDLE_DIR, "databricks.yml")).read()
    print(databricks_yml)
except Exception as e:
    databricks_yml = None
    print("Could not read the file from here — open materials/cicd/databricks.yml in the workspace file browser instead.")
    print(e)

## Task 1 — Inspect `databricks.yml`: bundle anatomy

Read the bundle root file printed above (or open `materials/cicd/databricks.yml`) and fill in the
`bundle_structure` dict describing what you found.

**What you need to do:** fill in the bundle name, the default target, that target's mode, and the
list of declared variable names.

In [0]:
# TODO: Fill in the dict by reading databricks.yml (printed in the Setup cell)
bundle_structure = {
    "bundle_name":         ...,   # bundle.name
    "default_target":      ...,   # which target has default: true?
    "default_target_mode": ...,   # that target's mode
    "variable_names":      [...], # names of ALL declared variables
}
print(bundle_structure)

#### Hint — Task 1

A bundle root file (`databricks.yml`) has four top-level areas you should be able to read cold on the exam:

- **`bundle:`** — the bundle `name`, namespacing everything the bundle deploys.
- **`include:`** — additional YAML files merged into the config (here: `resources/*.yml`, one file per resource).
- **`variables:`** — declared inputs with optional `default`s; referenced as `${var.<name>}` and overridden per deploy with `--var="name=value"`.
- **`targets:`** — named environments. Exactly one can carry `default: true`; `mode: development` gives dev-loop behavior (name prefixes, paused triggers), `mode: production` is strict.

Also note `sync.paths` — this bundle pulls in source folders that live *outside* the bundle root
(`../lakeflow/lakeflow_demo`, `../orchestration`), so `bundle deploy` uploads them too.

In [0]:
# -- Validation --
assert bundle_structure["bundle_name"] == "retailhub", "Check bundle.name in databricks.yml"
assert bundle_structure["default_target"] == "dev", "Which target carries default: true?"
assert bundle_structure["default_target_mode"] == "development", "Check the mode of the dev target"
assert sorted(bundle_structure["variable_names"]) == ["catalog", "schema_prefix"], \
    f"Expected the two declared variables, got: {bundle_structure['variable_names']}"
print("Task 1 OK: bundle anatomy understood")

## Task 2 — Compose the deploy command

The bundle's `catalog` variable defaults to `retailhub_trainer` — deploying without overriding it
would target the trainer's catalog (and fail on permissions). Compose the exact deploy command
**for your own catalog** as a Python string.

**What you need to do:** build `deploy_command` — a `databricks bundle deploy` invocation targeting
the `dev` target and overriding the `catalog` variable with **your** `CATALOG` value.

In [0]:
# TODO: Compose the deploy command using YOUR catalog (the CATALOG variable)
# Shape: databricks bundle deploy -t dev --var="catalog=<your catalog>"
deploy_command = ...  # YOUR CODE HERE (f-string)

print(deploy_command)

#### Hint — Task 2

**Command shape**

```
databricks bundle deploy -t <target> --var="<name>=<value>"
```

- `-t dev` selects the target (it is the default here, but being explicit is good CI hygiene).
- `--var="catalog=retailhub_jan_kowalski"` overrides a declared variable for this invocation.
  Multiple variables → repeat the flag.
- Your catalog name is already in the `CATALOG` variable exported by `00_setup` — use an f-string.

The same variable can also be set via environment (`BUNDLE_VAR_catalog`) or per-target
`variables:` overrides in `databricks.yml` — the `--var` flag wins over defaults.

In [0]:
# -- Validation --
assert isinstance(deploy_command, str), "deploy_command must be a string"
assert deploy_command.startswith("databricks bundle deploy"), "Start with: databricks bundle deploy"
assert "-t dev" in deploy_command, "Target the dev target explicitly: -t dev"
assert f"catalog={CATALOG}" in deploy_command, "Override the catalog variable with YOUR catalog (use CATALOG)"
print(f"Task 2 OK: {deploy_command}")

## Optional — Git folder workflow before deploying (objective 5.1)

> 👨‍🏫 **Trainer-dependent.** Needs a Git folder connected to a Git provider where you can push (Git credentials set in **Settings → Linked accounts**). If the provider is not connected, read the steps and continue with Task 3 — nothing below is checked by an assert.

1. **Workspace** → your Git folder → click the **branch name** → Git dialog → **Create Branch** `feature/<your_name>` from `main`.
2. Open `materials/cicd/resources/retailhub_job.yml` and make a harmless change, e.g. extend the `description` or add
   ```yaml
         tags:
           owner: <your_name>
   ```
   (at the same indentation as `max_concurrent_runs`).
3. Git dialog → check the diff → commit message `lab09: tag retailhub_job` → **Commit & Push**.
4. In the Git provider open a **pull request** `feature/<your_name>` → `main` (do not merge unless the trainer says so).
5. Stay on your feature branch: Task 3 deploys **the files you run it from** — the Git folder (web terminal / workspace UI) or your local clone after pulling `feature/<your_name>` — so your change ends up in the deployed job — a mini CI/CD loop.


## Task 3 — Validate & deploy the bundle

> 🖥️ **Deployment happens outside this notebook.** Pick ONE option from the table below, follow it,
> then record what you did in the answer cell — the check cell verifies the deployment in your workspace.

| Your situation | Option | Value for `how_i_ran_it` |
|---|---|---|
| Databricks CLI installed on your laptop | **A** — Local Databricks CLI | `"local_cli"` |
| Your workspace offers a web terminal (check with the trainer) | **B** — Web terminal | `"web_terminal"` |
| No CLI, but `databricks.yml` in your Git folder shows a **Deploy bundle** button | **C** — Workspace UI deploy | `"workspace_ui"` |
| None of the above | **D** — Trainer-driven fallback | `"trainer"` |

### Option A — Local Databricks CLI (recommended — tested with CLI v1.16.1)
1. Install the current Databricks CLI (see `09_cicd_and_automation` → *Databricks CLI — Setup*; **not** `pip install databricks-cli`) and check `databricks -v`.
2. Clone the repo locally (same branch as your Git folder), authenticate once, then validate and deploy from the bundle root.
   Replace `<YOUR CATALOG>` with your catalog — or paste the command printed by **Task 2** and append `-p TRAINING`
   (`validate` takes the same flags):

```bash
databricks auth login --host <workspace-url> --profile TRAINING
cd <your clone>/materials/cicd

databricks bundle validate -t dev --var="catalog=<YOUR CATALOG>" -p TRAINING
# read the summary: name, target, workspace host, user, path

databricks bundle deploy   -t dev --var="catalog=<YOUR CATALOG>" -p TRAINING   # <- Task 2 command (+ profile)
```

### Option B — Web terminal (only if available on your workspace)
The web terminal needs compute with the terminal enabled — it is **not verified** on this serverless-first
training workspace, so check with the trainer first. If you have it: open it (compute attached to the notebook → **Terminal**,
or the terminal icon in the bottom panel), `cd` into the bundle root inside your Git folder and run the same two commands
(authentication is normally inherited, so no `-p` profile) — the command printed by **Task 2** works as-is:

```bash
cd /Workspace/Users/<your login>/<git folder name>/materials/cicd
databricks bundle validate -t dev --var="catalog=<YOUR CATALOG>"
databricks bundle deploy   -t dev --var="catalog=<YOUR CATALOG>"
```

### Option C — Workspace UI deploy (no CLI)
If your workspace shows a **Deploy bundle** button when you open `materials/cicd/databricks.yml`
in your Git folder, you can validate/deploy from the workspace UI instead (make sure `catalog` resolves to **your** catalog)
and record `"workspace_ui"`. Availability varies between workspaces — ask the trainer.

### Option D — 👨‍🏫 Trainer-driven fallback (no CLI available)
The trainer runs `validate` + `deploy` on the shared screen. **Follow with this observation checklist:**

- [ ] `validate` prints the bundle **name** (`retailhub`) and **target** (`dev`)
- [ ] `validate` prints the **workspace path** the bundle will deploy to (`/Workspace/Users/<user>/.bundle/retailhub/dev`)
- [ ] `validate` ends with **no errors** ("Validation OK")
- [ ] `deploy` uploads files then reports **deployment complete**
- [ ] In **Jobs & Pipelines** a job **`[dev <user>] retailhub_job`** and a pipeline **`[dev <user>] retailhub_pipeline`** appeared
- [ ] The `[dev …]` prefix comes from `mode: development` — `retailhub_job` has no schedule (it is commented out in the YAML), but dev mode would pause any schedule/trigger

Then answer Task 4/5 against the trainer's deployed job (you have view access).

In [0]:
# TODO: Record how it went (this is your lab log — be honest, the assert checks it)
task3_result = {
    "how_i_ran_it":       ...,  # "local_cli" | "web_terminal" | "workspace_ui" | "trainer"
    "validate_succeeded": ...,  # did `bundle validate -t dev` finish without errors? True/False
}
print(task3_result)

In [0]:
# -- Validation --
from databricks.sdk import WorkspaceClient

assert task3_result["how_i_ran_it"] in ("local_cli", "web_terminal", "workspace_ui", "trainer"), \
    "Use one of: local_cli, web_terminal, workspace_ui, trainer"
assert task3_result["validate_succeeded"] is True, \
    "bundle validate must pass before deploying — ask the trainer if it failed"

# Real check: a dev-mode deploy writes its files and state under YOUR user home
w = WorkspaceClient()
_user_name = w.current_user.me().user_name
bundle_root_path = f"/Workspace/Users/{_user_name}/.bundle/retailhub/dev"
try:
    w.workspace.get_status(bundle_root_path)
    bundle_deployed, _err = True, None
except Exception as e:   # NotFound when nothing was deployed under your identity
    bundle_deployed, _err = False, e

if task3_result["how_i_ran_it"] == "trainer":
    print(f"Trainer-driven deploy — your own bundle folder {'exists' if bundle_deployed else 'is not expected'}: {bundle_root_path}")
    print("Task 3 OK (trainer-driven): Task 4 looks up the trainer's deployed job")
else:
    assert bundle_deployed, (
        f"No bundle deployment found at {bundle_root_path}. Run `databricks bundle deploy -t dev "
        f'--var="catalog={CATALOG}"` from materials/cicd as the same user as this notebook, then re-run this cell. '
        f"(details: {_err})"
    )
    print(f"Task 3 OK: bundle deployed via {task3_result['how_i_ran_it']} — found {bundle_root_path}")

## Task 4 — Verify the deployment from this notebook

Prove the deploy worked **programmatically** — no UI clicking. Use the Databricks SDK
(`databricks.sdk`, pre-installed on Databricks compute) to list jobs and find the deployed
RetailHub job.

**What you need to do:** build `retailhub_jobs` — all jobs visible to you whose name contains
`retailhub_job`. Remember: `mode: development` deployed it as **`[dev <user>] retailhub_job`**.

Run the check cell — it selects `retailhub_job_id` used in Task 5.

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# TODO: list all jobs you can see whose name contains "retailhub_job"
retailhub_jobs = []  # YOUR CODE HERE — replace [] with a list comprehension over w.jobs.list(), filtering on j.settings.name

for j in retailhub_jobs:
    print(f"{j.job_id}  {j.settings.name}")

#### Hint — Task 4

**WorkspaceClient without arguments**
Inside a Databricks notebook, `WorkspaceClient()` authenticates automatically from the runtime context
— no host/token needed.

**Listing jobs**

```python
for j in w.jobs.list():
    print(j.job_id, j.settings.name)
```

`w.jobs.list()` returns an iterator of `BaseJob`; the display name is `j.settings.name`.
A list comprehension with a substring test on the name is all you need.

**Why the `[dev …]` prefix?** `mode: development` prefixes every resource name with
`[dev <your user name>]` so many people can deploy the same bundle into one workspace
without collisions. In `mode: production` the name stays exactly `retailhub_job`.

In [0]:
# -- Validation --
assert len(retailhub_jobs) >= 1, "No deployed retailhub_job found — did Task 3 deploy succeed?"
_names = [j.settings.name for j in retailhub_jobs]
assert any(n.startswith("[dev") and "retailhub_job" in n for n in _names), \
    f"Expected a dev-mode name like '[dev <user>] retailhub_job', got: {_names}"

# Prefer your own copy if several participants deployed into this workspace
_me = w.current_user.me().user_name.split("@")[0].replace(".", "_").lower()
_mine = [j for j in retailhub_jobs if _me in j.settings.name.lower()]
if not _mine:
    print(f"WARNING: no retailhub_job with '{_me}' in its name — falling back to '{retailhub_jobs[0].settings.name}', "
          "which may be another participant's (or the trainer's) deployment")
    _mine = retailhub_jobs
retailhub_job_id = _mine[0].job_id
print(f"Task 4 OK: found {_names} — using job_id={retailhub_job_id}")

## Task 5 — Run the job and confirm it succeeded

Trigger a run of the deployed job, then confirm from this notebook that it reached a **terminal
SUCCESS state** via the SDK.

**Start the run — pick one:**

```bash
# Databricks CLI (local, or web terminal if available) — streams task states until the job finishes.
# Local CLI: keep -p TRAINING (as in Task 3, Option A). Web terminal: drop -p TRAINING.
databricks bundle run -t dev --var="catalog=<YOUR CATALOG>" -p TRAINING retailhub_job
```

- …or **Jobs & Pipelines → `[dev …] retailhub_job` → Run now** (allowed — the point of the lab is the bundle, not avoiding the UI),
- …or 👨‍🏫 the **trainer** runs the shared deployment and you verify that run.

The run goes through validate → pipeline refresh → report. Re-run **both** cells below until `result_state` is no longer `None`.

In [0]:
# TODO: fetch the latest run of retailhub_job_id and extract its result state
from itertools import islice

# list_runs() pages lazily through ALL runs of the job — take only the newest 5 (newest first)
runs = list(islice(w.jobs.list_runs(job_id=retailhub_job_id, limit=5), 5))

latest_run   = ...   # YOUR CODE HERE — the most recent run (None if runs is empty)
result_state = ...   # YOUR CODE HERE — its result state as a string ("SUCCESS", ...)
                     # hint: latest_run.state.result_state, None until the run terminates

print(f"life_cycle_state = {latest_run.state.life_cycle_state.value if latest_run and latest_run.state else '?'}")
print(f"result_state     = {result_state}")

#### Hint — Task 5

**Fetching runs for one job**

`runs` (fetched in the cell above) is a plain list, **newest first** — guard for an empty list before indexing it.

**Run state anatomy** — two different fields, a classic exam trap:
- `run.state.life_cycle_state` — where the run is in its lifecycle: `PENDING → RUNNING → TERMINATED`
- `run.state.result_state` — **only set once terminated**: `SUCCESS`, `FAILED`, `CANCELED`, `TIMEDOUT`

So: a healthy finished run has `life_cycle_state = TERMINATED` **and** `result_state = SUCCESS`.
Both are enums — use `.value` to get the string, and guard for `None` while the run is still going.

In [0]:
# -- Validation --
assert len(runs) >= 1, "No runs found — trigger the job first (bundle run / Run now / trainer)"
assert result_state is not None, \
    "Run has not reached a terminal state yet — wait a bit and re-run the previous cell + this one"
assert str(result_state).upper() == "SUCCESS", \
    f"Run terminated with {result_state} — open the run in Jobs & Pipelines and check the failed task"
print(f"Task 5 OK: run {latest_run.run_id} finished with result_state=SUCCESS")

## Task 6 — Reflection: `dev` vs `prod` targets

Look at the commented `prod` stub at the bottom of `databricks.yml` and at what `mode: development`
did to your deployment, then fill in the reflection dict.

> `retailhub_job` declares no schedule (it is commented out in `resources/retailhub_job.yml`), so you cannot *see* a paused
> trigger in this deployment — the question is what `mode: development` **would** do to any schedule or trigger.

In [0]:
# TODO: fill in the reflection
reflection = {
    "dev_name_prefix":       ...,  # string: what development mode prepends to resource names (look at the job name in Jobs & Pipelines)
    "dev_pauses_triggers":   ...,  # are schedules/triggers paused in a development-mode deployment? True/False
    "prod_runs_as":          ...,  # recommended identity for prod deployments: "personal_user" | "service_principal"
    "promotion_changes":     ...,  # what changes when promoting dev -> prod: "the_code" | "only_target_config"
}
print(reflection)

#### Hint — Task 6

Compare what you saw with what the `prod` stub declares:

| Aspect | `dev` (`mode: development`) | `prod` (`mode: production`) |
|---|---|---|
| Resource names | prefixed | exact names from YAML |
| Schedules & triggers | paused automatically | active |
| Deploy path | your user home (`/Users/<you>/.bundle/...`) | a restricted folder — the service principal's (`run_as`) home, not `/Workspace/Shared` (writable by all users) |
| Identity | you | typically a **service principal** (`run_as`) |
| Code | **the same YAML + sources** | the same — only target config differs |

The last row is the core CI/CD idea (and an exam favorite): promotion between environments changes
**target configuration**, never the code.

In [0]:
# -- Validation --
assert "[dev" in str(reflection["dev_name_prefix"]), \
    "Look at your deployed job's name in Jobs & Pipelines"
assert reflection["dev_pauses_triggers"] is True, \
    "development mode pauses schedules/triggers so dev copies never fire on their own"
assert reflection["prod_runs_as"] == "service_principal", \
    "Production deployments should not depend on a person's identity"
assert reflection["promotion_changes"] == "only_target_config", \
    "Same YAML + sources deploy everywhere; only the target section differs"
print("Task 6 OK: dev vs prod understood")

## Summary

| Task | Topic | Key Point |
|------|-------|-----------|
| 1 | Bundle anatomy | `bundle` / `include` / `variables` / `targets` (+ `sync.paths` for outside sources) |
| 2 | Variables | `${var.catalog}` in YAML, overridden per deploy with `--var="catalog=..."` |
| 3 | CLI flow | `validate` → `deploy` → resources appear; `mode: development` = `[dev …]` prefix + paused triggers |
| 4 | SDK verification | `WorkspaceClient().jobs.list()` — verify deployments programmatically, not by clicking |
| 5 | Run states | `life_cycle_state` (PENDING/RUNNING/TERMINATED) vs `result_state` (SUCCESS/FAILED/…) |
| 6 | dev vs prod | Promotion changes target config (identity, paths, variables) — never the code |

**Stretch goals (optional):**
- Re-point the `publish_report` task at a RetailHub gold table (edit `resources/retailhub_job.yml`
  — note it needs a RetailHub-aware report notebook, see the comment in the YAML) and redeploy.
- Tear your deployment down: `databricks bundle destroy -t dev --var="catalog=<YOUR CATALOG>" -p TRAINING` (web terminal: without `-p TRAINING`)
  — then check Jobs & Pipelines to confirm both resources are gone.

> 🎯 **Exam (Implementing CI/CD):** Git folder branch/commit/push/PR flow, bundle file anatomy, `validate`/`deploy`/`run`/`destroy`,
> targets & modes, variable overrides, and why bundles beat manual UI configuration
> (versioned, reviewable, repeatable, environment-promotable).

← [09 — CI/CD & Automation](../demo/09_cicd_and_automation.ipynb) | **[README](../../../README.md)** | [Lab — Troubleshooting →](lab_troubleshooting.ipynb)